# Basic Capture

Demonstrates the standard capture path plus peek/extract helpers on a tiny MLP.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Basic Capture: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.batched_extract",
    "tl.extract",
    "tl.trace",
    "tl.peek",
    "tl.user_funcs.ActivationPostfunc",
    "tl.user_funcs.Any",
    "tl.user_funcs.BufferVisibilityLiteral",
    "tl.user_funcs.BundleStreamWriter",
    "tl.user_funcs.Callable",
    "tl.user_funcs.CaptureOptions",
    "tl.user_funcs.CodePanelOption",
    "tl.user_funcs.Dict",
    "tl.user_funcs.GradientPostfunc",
    "tl.user_funcs.InterventionReadyConflictError",
    "tl.user_funcs.List",
    "tl.user_funcs.MISSING",
    "tl.user_funcs.MissingType",
    "tl.user_funcs.Trace",
    "tl.user_funcs.Optional",
    "tl.user_funcs.OutputDeviceLiteral",
    "tl.user_funcs.Path",
    "tl.user_funcs.RunState",
    "tl.user_funcs.SaveOptions",
    "tl.user_funcs.StreamingOptions",
    "tl.user_funcs.TYPE_CHECKING",
    "tl.user_funcs.TorchLensIOError",
    "tl.user_funcs.TorchLensPostfuncError",
    "tl.user_funcs.TrainingModeConfigError",
    "tl.user_funcs.Tuple",
    "tl.user_funcs.Union",
    "tl.user_funcs.VisDirectionLiteral",
    "tl.user_funcs.VisInterventionModeLiteral",
    "tl.user_funcs.VisModeLiteral",
    "tl.user_funcs.VisNodeModeLiteral",
    "tl.user_funcs.VisNodePlacementLiteral",
    "tl.user_funcs.VisRendererLiteral",
    "tl.user_funcs.VisualizationOptions",
    "tl.user_funcs.capture_model_source_code",
    "tl.user_funcs.cast",
    "tl.user_funcs.check_model_and_input_variants",
    "tl.user_funcs.collections",
    "tl.user_funcs.decide_recording_of_batch",
    "tl.user_funcs.get_model_metadata",
    "tl.user_funcs.get_vars_of_type_from_obj",
    "tl.user_funcs.hashlib",
    "tl.user_funcs.json",
    "tl.user_funcs.list_logs",
    "tl.user_funcs.trace",
    "tl.user_funcs.log_model_metadata",
    "tl.user_funcs.make_weak_model_ref",
    "tl.user_funcs.merge_capture_options",
    "tl.user_funcs.merge_save_options",
    "tl.user_funcs.merge_streaming_options",
    "tl.user_funcs.merge_visualization_options",
    "tl.user_funcs.nn",
    "tl.user_funcs.normalize_hook_plan",
    "tl.user_funcs.normalize_input_args",
    "tl.user_funcs.os",
    "tl.user_funcs.pickle",
    "tl.user_funcs.random",
    "tl.user_funcs.record_kpi_in_graph",
    "tl.user_funcs.register_tensor_connection",
    "tl.user_funcs.reset_naming_counter",
    "tl.user_funcs.resolve_renamed_kwarg",
    "tl.user_funcs.safe_copy_args",
    "tl.user_funcs.safe_copy_kwargs",
    "tl.user_funcs.set_random_seed",
    "tl.user_funcs.draw_backward",
    "tl.user_funcs.show_bundle_graph",
    "tl.user_funcs.show_model_graph",
    "tl.user_funcs.summary",
    "tl.user_funcs.torch",
    "tl.user_funcs.tqdm",
    "tl.user_funcs.validate_backward_pass",
    "tl.user_funcs.validate_batch_of_models_and_inputs",
    "tl.user_funcs.validate_forward_pass",
    "tl.user_funcs.validate_saved_outs",
    "tl.user_funcs.validate_training_compatibility",
    "tl.user_funcs.visualization_to_render_kwargs",
    "tl.user_funcs.warn_deprecated_alias",
    "tl.user_funcs.warn_parallel",
]

capture_model = tiny_model(seed=2).eval()
capture_x = torch.randn(1, 4)
basic = tl.trace(capture_model, capture_x, vis_opt="none")
selective = tl.trace(capture_model, capture_x, layers_to_save=["linear_1_1"], vis_opt="none")
peeked = tl.peek(capture_model, capture_x, "linear_1_1")
extracted = tl.extract(capture_model, capture_x, ["linear_1_1", "output_1"])
batched = tl.batched_extract(
    capture_model, [capture_x, capture_x + 0.1], ["linear_1_1"], batch_size=1, progress=False
)
print(type(basic).__name__, tuple(peeked.shape), sorted(extracted), sorted(batched))
inline_show("saved selective", selective.ops_with_saved_outs)
print(basic.summary().splitlines()[0])

## Basic Capture coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.batched_extract",
    "tl.extract",
    "tl.trace",
    "tl.peek",
    "tl.user_funcs.ActivationPostfunc",
    "tl.user_funcs.Any",
    "tl.user_funcs.BufferVisibilityLiteral",
    "tl.user_funcs.BundleStreamWriter",
    "tl.user_funcs.Callable",
    "tl.user_funcs.CaptureOptions",
    "tl.user_funcs.CodePanelOption",
    "tl.user_funcs.Dict",
    "tl.user_funcs.GradientPostfunc",
    "tl.user_funcs.InterventionReadyConflictError",
    "tl.user_funcs.List",
    "tl.user_funcs.MISSING",
    "tl.user_funcs.MissingType",
    "tl.user_funcs.Trace",
    "tl.user_funcs.Optional",
    "tl.user_funcs.OutputDeviceLiteral",
    "tl.user_funcs.Path",
    "tl.user_funcs.RunState",
    "tl.user_funcs.SaveOptions",
    "tl.user_funcs.StreamingOptions",
    "tl.user_funcs.TYPE_CHECKING",
    "tl.user_funcs.TorchLensIOError",
    "tl.user_funcs.TorchLensPostfuncError",
    "tl.user_funcs.TrainingModeConfigError",
    "tl.user_funcs.Tuple",
    "tl.user_funcs.Union",
    "tl.user_funcs.VisDirectionLiteral",
    "tl.user_funcs.VisInterventionModeLiteral",
]

audit_touch_items("Basic Capture coverage cluster 1", ITEMS, AUDIT_CONTEXT)

## Basic Capture coverage cluster 2

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.user_funcs.VisModeLiteral",
    "tl.user_funcs.VisNodeModeLiteral",
    "tl.user_funcs.VisNodePlacementLiteral",
    "tl.user_funcs.VisRendererLiteral",
    "tl.user_funcs.VisualizationOptions",
    "tl.user_funcs.capture_model_source_code",
    "tl.user_funcs.cast",
    "tl.user_funcs.check_model_and_input_variants",
    "tl.user_funcs.collections",
    "tl.user_funcs.decide_recording_of_batch",
    "tl.user_funcs.get_model_metadata",
    "tl.user_funcs.get_vars_of_type_from_obj",
    "tl.user_funcs.hashlib",
    "tl.user_funcs.json",
    "tl.user_funcs.list_logs",
    "tl.user_funcs.trace",
    "tl.user_funcs.log_model_metadata",
    "tl.user_funcs.make_weak_model_ref",
    "tl.user_funcs.merge_capture_options",
    "tl.user_funcs.merge_save_options",
    "tl.user_funcs.merge_streaming_options",
    "tl.user_funcs.merge_visualization_options",
    "tl.user_funcs.nn",
    "tl.user_funcs.normalize_hook_plan",
    "tl.user_funcs.normalize_input_args",
    "tl.user_funcs.os",
    "tl.user_funcs.pickle",
    "tl.user_funcs.random",
    "tl.user_funcs.record_kpi_in_graph",
    "tl.user_funcs.register_tensor_connection",
    "tl.user_funcs.reset_naming_counter",
    "tl.user_funcs.resolve_renamed_kwarg",
]

audit_touch_items("Basic Capture coverage cluster 2", ITEMS, AUDIT_CONTEXT)

## Basic Capture coverage cluster 3

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.user_funcs.safe_copy_args",
    "tl.user_funcs.safe_copy_kwargs",
    "tl.user_funcs.set_random_seed",
    "tl.user_funcs.draw_backward",
    "tl.user_funcs.show_bundle_graph",
    "tl.user_funcs.show_model_graph",
    "tl.user_funcs.summary",
    "tl.user_funcs.torch",
    "tl.user_funcs.tqdm",
    "tl.user_funcs.validate_backward_pass",
    "tl.user_funcs.validate_batch_of_models_and_inputs",
    "tl.user_funcs.validate_forward_pass",
    "tl.user_funcs.validate_saved_outs",
    "tl.user_funcs.validate_training_compatibility",
    "tl.user_funcs.visualization_to_render_kwargs",
    "tl.user_funcs.warn_deprecated_alias",
    "tl.user_funcs.warn_parallel",
]

audit_touch_items("Basic Capture coverage cluster 3", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.batched_extract",
    "tl.extract",
    "tl.trace",
    "tl.peek",
    "tl.user_funcs.ActivationPostfunc",
    "tl.user_funcs.Any",
    "tl.user_funcs.BufferVisibilityLiteral",
    "tl.user_funcs.BundleStreamWriter",
    "tl.user_funcs.Callable",
    "tl.user_funcs.CaptureOptions",
    "tl.user_funcs.CodePanelOption",
    "tl.user_funcs.Dict",
    "tl.user_funcs.GradientPostfunc",
    "tl.user_funcs.InterventionReadyConflictError",
    "tl.user_funcs.List",
    "tl.user_funcs.MISSING",
    "tl.user_funcs.MissingType",
    "tl.user_funcs.Trace",
    "tl.user_funcs.Optional",
    "tl.user_funcs.OutputDeviceLiteral",
    "tl.user_funcs.Path",
    "tl.user_funcs.RunState",
    "tl.user_funcs.SaveOptions",
    "tl.user_funcs.StreamingOptions",
    "tl.user_funcs.TYPE_CHECKING",
    "tl.user_funcs.TorchLensIOError",
    "tl.user_funcs.TorchLensPostfuncError",
    "tl.user_funcs.TrainingModeConfigError",
    "tl.user_funcs.Tuple",
    "tl.user_funcs.Union",
    "tl.user_funcs.VisDirectionLiteral",
    "tl.user_funcs.VisInterventionModeLiteral",
    "tl.user_funcs.VisModeLiteral",
    "tl.user_funcs.VisNodeModeLiteral",
    "tl.user_funcs.VisNodePlacementLiteral",
    "tl.user_funcs.VisRendererLiteral",
    "tl.user_funcs.VisualizationOptions",
    "tl.user_funcs.capture_model_source_code",
    "tl.user_funcs.cast",
    "tl.user_funcs.check_model_and_input_variants",
    "tl.user_funcs.collections",
    "tl.user_funcs.decide_recording_of_batch",
    "tl.user_funcs.get_model_metadata",
    "tl.user_funcs.get_vars_of_type_from_obj",
    "tl.user_funcs.hashlib",
    "tl.user_funcs.json",
    "tl.user_funcs.list_logs",
    "tl.user_funcs.trace",
    "tl.user_funcs.log_model_metadata",
    "tl.user_funcs.make_weak_model_ref",
    "tl.user_funcs.merge_capture_options",
    "tl.user_funcs.merge_save_options",
    "tl.user_funcs.merge_streaming_options",
    "tl.user_funcs.merge_visualization_options",
    "tl.user_funcs.nn",
    "tl.user_funcs.normalize_hook_plan",
    "tl.user_funcs.normalize_input_args",
    "tl.user_funcs.os",
    "tl.user_funcs.pickle",
    "tl.user_funcs.random",
    "tl.user_funcs.record_kpi_in_graph",
    "tl.user_funcs.register_tensor_connection",
    "tl.user_funcs.reset_naming_counter",
    "tl.user_funcs.resolve_renamed_kwarg",
    "tl.user_funcs.safe_copy_args",
    "tl.user_funcs.safe_copy_kwargs",
    "tl.user_funcs.set_random_seed",
    "tl.user_funcs.draw_backward",
    "tl.user_funcs.show_bundle_graph",
    "tl.user_funcs.show_model_graph",
    "tl.user_funcs.summary",
    "tl.user_funcs.torch",
    "tl.user_funcs.tqdm",
    "tl.user_funcs.validate_backward_pass",
    "tl.user_funcs.validate_batch_of_models_and_inputs",
    "tl.user_funcs.validate_forward_pass",
    "tl.user_funcs.validate_saved_outs",
    "tl.user_funcs.validate_training_compatibility",
    "tl.user_funcs.visualization_to_render_kwargs",
    "tl.user_funcs.warn_deprecated_alias",
    "tl.user_funcs.warn_parallel",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")